In [4]:
import pandas as pd
import numpy as np

file_path_Stones = '/Users/connorbrady/Desktop/Curling_files/Stones.csv'
file_path_Competition = '/Users/connorbrady/Desktop/Curling_files/Competition.csv'
file_path_Competitors = '/Users/connorbrady/Desktop/Curling_files/Competitors.csv'
file_path_Ends = '/Users/connorbrady/Desktop/Curling_files/Ends.csv'
file_path_Games = '/Users/connorbrady/Desktop/Curling_files/Games.csv'
file_path_Teams = '/Users/connorbrady/Desktop/Curling_files/Teams.csv'




stones = pd.read_csv(file_path_Stones)
competition = pd.read_csv(file_path_Competition)
competitors = pd.read_csv(file_path_Competitors)
ends = pd.read_csv(file_path_Ends)
games = pd.read_csv(file_path_Games)
teams = pd.read_csv(file_path_Teams)






Here were replace out of bound shots and not thrown yet shots to NaN's

In [5]:
import numpy as np

stone_cols = [c for c in stones.columns if c.startswith("stone_")]
stones[stone_cols] = stones[stone_cols].replace({0: np.nan, 4095: np.nan})


giving an order to the shots in an end
1st - Competition
2nd - Session
3rd - Game
4th - Ends
5th - Shot

In [6]:
stones_sorted = stones.sort_values(
    ["CompetitionID", "SessionID", "GameID", "EndID", "ShotID"]
)
# Add the type of short order 
stones_sorted["shot_order"] = (
    stones_sorted
    .groupby(["CompetitionID", "SessionID", "GameID", "EndID"])
    .cumcount() + 1
)


create an array with one each row being the end, the team who threw the third shot, and the full board snapshot after that shot

In [7]:
state_after_3 = stones_sorted[stones_sorted["shot_order"] == 3].copy()


attach points scored by this team to this end

In [8]:
state_with_label = state_after_3.merge(
    ends[["CompetitionID", "SessionID", "GameID", "EndID", "TeamID", "Result"]],
    on=["CompetitionID", "SessionID", "GameID", "EndID", "TeamID"],
    how="left"
)


In [9]:
assert state_with_label["Result"].isna().sum() == 0


so now each row follows the logic-Given this board state after Shot 3, this team eventually scored Result points in the end.

In [10]:
print((state_with_label.head))

<bound method NDFrame.head of       CompetitionID  SessionID  GameID  EndID  ShotID  TeamID  PlayerID  Task  \
0                 0          1       1      1       9      19         2     0   
1                 0          1       1      2       9      27         2     0   
2                 0          1       1      3       9      19         2     0   
3                 0          1       1      4       9      27         2     0   
4                 0          1       1      5       9      19         2     0   
...             ...        ...     ...    ...     ...     ...       ...   ...   
2632       24250026         49       1      3       9      14         2     0   
2633       24250026         49       1      4       9      24         2     0   
2634       24250026         49       1      5       9      24         2     0   
2635       24250026         49       1      6       9      24         2     0   
2636       24250026         49       1      7       9      14         2     3  

In [11]:
keep_cols = [
    "CompetitionID", "SessionID", "GameID", "EndID",
    "TeamID", "Result"
] + stone_cols

model_df = state_with_label[keep_cols]

In [12]:
model_df["Result"].value_counts().sort_index()


Result
0    1819
1     528
2     167
3      55
4       7
5       1
9      60
Name: count, dtype: int64

In [13]:
model_df.duplicated(
    ["CompetitionID", "SessionID", "GameID", "EndID", "TeamID"]
).sum()

np.int64(0)

In the next two code segemnts we fix the problem of 4 5 and 9 cases grouping them in with 3 points ends. so every score above 3 gets grouped together. 

In [14]:
def bin_result(x):
    if x >= 3:
        return 3
    else:
        return x

model_df["Result_binned"] = model_df["Result"].apply(bin_result)


/var/folders/jn/t7pxmvwx0gl6ypytq5y4j0340000gn/T/ipykernel_27517/1098613871.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["Result_binned"] = model_df["Result"].apply(bin_result)


In [15]:
model_df["Result_binned"].value_counts().sort_index()


Result_binned
0    1819
1     528
2     167
3     123
Name: count, dtype: int64

Here we are just doing a pretty basic check to see if all our labels are created and printing any missing ones. None appear so we can continue.

In [16]:
assert "Result" in model_df.columns
missing_labels = model_df["Result"].isna().sum()
print("Missing labels:", missing_labels)
assert missing_labels == 0

Missing labels: 0


Now we check if there are any weird numbers

In [17]:
print("Result dtype:", model_df["Result"].dtype)
print("Unique Result values:", sorted(model_df["Result"].unique()))


Result dtype: int64
Unique Result values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(9)]


Here were just creating target bins called Result_binned to our model can distinguish between different scored ends easier

In [18]:
# Make sure Result is numeri
model_df["Result"] = pd.to_numeric(model_df["Result"], errors="coerce")
assert model_df["Result"].isna().sum() == 0, "Result had non-numeric values that became NaN."

# Create binned target: 0,1,2,3
model_df["Result_binned"] = np.where(model_df["Result"] >= 3, 3, model_df["Result"]).astype(int)


/var/folders/jn/t7pxmvwx0gl6ypytq5y4j0340000gn/T/ipykernel_27517/1703128258.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["Result"] = pd.to_numeric(model_df["Result"], errors="coerce")
/var/folders/jn/t7pxmvwx0gl6ypytq5y4j0340000gn/T/ipykernel_27517/1703128258.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["Result_binned"] = np.where(model_df["Result"] >= 3, 3, model_df["Result"]).astype(int)


Here we just check the distribution of the scoring bins and make sure they look accurate

In [20]:
print("Raw Result distribution:")
print(model_df["Result"].value_counts().sort_index())

print("\nBinned Result distribution (0/1/2/3+):")
print(model_df["Result_binned"].value_counts().sort_index())


Raw Result distribution:
Result
0    1819
1     528
2     167
3      55
4       7
5       1
9      60
Name: count, dtype: int64

Binned Result distribution (0/1/2/3+):
Result_binned
0    1819
1     528
2     167
3     123
Name: count, dtype: int64


Make sure we dont have any remaining bins exept 1, 2, or 3

In [21]:
allowed = {0, 1, 2, 3}
observed = set(model_df["Result_binned"].unique())
print("Observed binned label set:", observed)
assert observed.issubset(allowed), f"Unexpected labels found: {observed - allowed}"


Observed binned label set: {np.int64(0), np.int64(1), np.int64(2), np.int64(3)}


Here were taking into account that zreo is by fat going to be the most common score in an end. We dont want the model to hallucinte and get tricked by this increase in0's

In [22]:
class_counts = model_df["Result_binned"].value_counts().sort_index()
class_weights = (class_counts.sum() / (len(class_counts) * class_counts)).to_dict()

print("Class counts:")
print(class_counts)
print("\nSuggested inverse-frequency class weights:")
print(class_weights)


Class counts:
Result_binned
0    1819
1     528
2     167
3     123
Name: count, dtype: int64

Suggested inverse-frequency class weights:
{0: 0.3624244090159428, 1: 1.2485795454545454, 2: 3.947604790419162, 3: 5.359756097560975}


Here were gonna move on to establishgin distances from stones to the button. since the button is a (750, 800) we can use a simple distance equation to track how far different stones are from the button 

our first code chund just sets up the dist function so we can call the distance to the button anytime we want

In [23]:
BUTTON_X, BUTTON_Y = 750.0, 800.0

def euclid_dist(x,y, x0 = BUTTON_X, y0 = BUTTON_Y):
    return np.sqrt((x-x0)**2 + (y-y0)**2)

In [26]:
# We'll need games_df (your Games.csv dataframe) loaded already as `games`
needed_cols = ["CompetitionID", "SessionID", "GameID", "TeamID1", "TeamID2"]
games_key = games[needed_cols].drop_duplicates()

# Merge TeamID1/TeamID2 into model_df so each row knows the two teams in the match
model_df = model_df.merge(
    games_key,
    on=["CompetitionID", "SessionID", "GameID"],
    how="left"
)

# Sanity checks: make sure merge worked
assert model_df["TeamID1"].isna().sum() == 0, "Some rows missing TeamID1 after merge."
assert model_df["TeamID2"].isna().sum() == 0, "Some rows missing TeamID2 after merge."


In [27]:
bad_rows = ~model_df["TeamID"].isin(model_df[["TeamID1", "TeamID2"]].values.ravel())
# The line above is a bit strict; better:
bad_rows = ~model_df.apply(lambda r: r["TeamID"] in (r["TeamID1"], r["TeamID2"]), axis=1)

print("Rows where TeamID is not TeamID1 or TeamID2:", bad_rows.sum())
assert bad_rows.sum() == 0, "Found rows where TeamID isn't one of the game teams."


Rows where TeamID is not TeamID1 or TeamID2: 0


Here were gonna use the TeamID1 and TeamID2 we defined to try to sort shots into team1 or team2. 

In [28]:
def stone_xy_cols(indices):
    """Return lists of x and y column names for a set of stone indices."""
    x_cols = [f"stone_{i}_x" for i in indices]
    y_cols = [f"stone_{i}_y" for i in indices]
    return x_cols, y_cols

TEAM1_STONES = list(range(1, 7))   # stones 1-6
TEAM2_STONES = list(range(7, 13))  # stones 7-12

team1_x, team1_y = stone_xy_cols(TEAM1_STONES)
team2_x, team2_y = stone_xy_cols(TEAM2_STONES)


Now lets write the code to actually get distances for team 1 and team 2 stones. Numpy is gonna do some heavy lifting here.

In [29]:
#Compute distances for 1-6
import numpy as np
team1_dists = []
for i in TEAM1_STONES:
    dx = euclid_dist(model_df[f"stone_{i}_x"], model_df[f"stone_{i}_y"])
    team1_dists.append(dx)
team1_dists = np.vstack(team1_dists).T

#Compute distances for 7-12
team2_dists = []
for i in TEAM2_STONES:
    dx = euclid_dist(model_df[f"stone_{i}_x"], model_df[f"stone_{i}_y"])
    team2_dists.append(dx)
team2_dists = np.vstack(team2_dists).T  # shape: (n_rows, 6)

#ignore Nans
team1_min = np.nanmin(team1_dists, axis=1)
team2_min = np.nanmin(team2_dists, axis=1)

is_team1 = (model_df["TeamID"] == model_df["TeamID1"]).values

model_df["our_closest_dist"] = np.where(is_team1, team1_min, team2_min)
model_df["opp_closest_dist"] = np.where(is_team1, team2_min, team1_min)

Here were gonan cover edge cases and replace Nan values for stones that arent yet in place with a awsomely large number

In [30]:
# Replace inf with a large distance (bigger than any realistic distance on sheet)
LARGE_DIST = 1e6
model_df["our_closest_dist"] = model_df["our_closest_dist"].replace([np.inf, -np.inf], np.nan).fillna(LARGE_DIST)
model_df["opp_closest_dist"] = model_df["opp_closest_dist"].replace([np.inf, -np.inf], np.nan).fillna(LARGE_DIST)


Here were just gonna set a radius for whats in the "House"

In [31]:
HOUSE_R = 250.0

using the house radius value we can compute how many of our stones are in the house and how many of the openents stones are in the distance

In [32]:
team1_in_house = np.sum(team1_dists <= HOUSE_R, axis=1)
team2_in_house = np.sum(team2_dists <= HOUSE_R, axis=1)

model_df["our_stones_in_house"] = np.where(is_team1, team1_in_house, team2_in_house).astype(int)
model_df["opp_stones_in_house"] = np.where(is_team1, team2_in_house, team1_in_house).astype(int)


Now this works out great as we can easily calculate who has control of the house by seeing if we or if our oponents stones are closer to the button. well define this as "house control"

In [33]:
model_df["house_control_diff"] = model_df["our_stones_in_house"] - model_df["opp_stones_in_house"]


Now were going to do a check to see if our system is working right. To do this we just collect the distribution of the distant distances our our stones to the button. so far everything looks correct

In [37]:
print(model_df[["our_closest_dist", "opp_closest_dist", "our_stones_in_house", "opp_stones_in_house", "house_control_diff"]].describe())

print("\nMean features by Result_binned:")
print(model_df.groupby("Result_binned")[["our_closest_dist", "opp_closest_dist", "our_stones_in_house", "opp_stones_in_house", "house_control_diff"]].mean())


       our_closest_dist  opp_closest_dist  our_stones_in_house  \
count       2637.000000       2637.000000          2637.000000   
mean         162.531941        169.525485             1.239666   
std          168.765456        123.394561             0.760992   
min            3.162278          4.123106             0.000000   
25%           65.490457         75.006666             1.000000   
50%           96.607453        150.000000             1.000000   
75%          202.607996        200.242353             2.000000   
max         1166.352005        608.935136             2.000000   

       opp_stones_in_house  house_control_diff  
count          2637.000000         2637.000000  
mean              1.287448           -0.047782  
std               0.797405            0.717751  
min               0.000000           -2.000000  
25%               1.000000            0.000000  
50%               2.000000            0.000000  
75%               2.000000            0.000000  
max          

Now were doing to merge our powerplay variable into our dataframe.

In [38]:
# Bring PowerPlay into model_df 
pp_key = ends[["CompetitionID", "SessionID", "GameID", "EndID", "TeamID", "PowerPlay"]].copy()

model_df = model_df.merge(
    pp_key,
    on=["CompetitionID", "SessionID", "GameID", "EndID", "TeamID"],
    how="left"
)

# If not using PP, PowerPlay will be NaN
model_df["pp_used"] = model_df["PowerPlay"].fillna(0).astype(int).clip(0, 2)
model_df["pp_used"] = (model_df["pp_used"] > 0).astype(int)


In [39]:
print(model_df["pp_used"].value_counts())


pp_used
0    2637
Name: count, dtype: int64


Create different sides for our powerplays

In [40]:
# Create pp_side: 0 none, 1 right, 2 left
model_df["pp_side"] = model_df["PowerPlay"].fillna(0).astype(int).clip(0, 2)

# One-hot style indicators
model_df["pp_right"] = (model_df["pp_side"] == 1).astype(int)
model_df["pp_left"]  = (model_df["pp_side"] == 2).astype(int)


In [41]:
print(model_df[["pp_side", "pp_right", "pp_left"]].value_counts().sort_index())


pp_side  pp_right  pp_left
0        0         0          2637
Name: count, dtype: int64


In [42]:
print("Ends PowerPlay value counts (raw):")
print(ends["PowerPlay"].value_counts(dropna=False).head(10))

print("\nNon-null PowerPlay rows:", ends["PowerPlay"].notna().sum())


Ends PowerPlay value counts (raw):
PowerPlay
NaN    4676
1.0     313
2.0     285
Name: count, dtype: int64

Non-null PowerPlay rows: 598


In [43]:
keys = ["CompetitionID","SessionID","GameID","EndID","TeamID"]

print("model_df key dtypes:")
print(model_df[keys].dtypes)

print("\nends key dtypes:")
print(ends[keys].dtypes)


model_df key dtypes:
CompetitionID    int64
SessionID        int64
GameID           int64
EndID            int64
TeamID           int64
dtype: object

ends key dtypes:
CompetitionID    int64
SessionID        int64
GameID           int64
EndID            int64
TeamID           int64
dtype: object


In [44]:
for df in [model_df, ends]:
    for c in ["CompetitionID","SessionID","GameID","EndID","TeamID"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")


In [45]:
# Drop old columns if they exist to avoid confusion
for col in ["PowerPlay", "pp_used", "pp_side", "pp_right", "pp_left"]:
    if col in model_df.columns:
        model_df = model_df.drop(columns=[col])

pp_key = ends[["CompetitionID","SessionID","GameID","EndID","TeamID","PowerPlay"]].copy()

model_df = model_df.merge(
    pp_key,
    on=["CompetitionID","SessionID","GameID","EndID","TeamID"],
    how="left"
)


In [46]:
print("After merge: non-null PowerPlay rows in model_df:", model_df["PowerPlay"].notna().sum())
print("Unique PowerPlay values in model_df:", pd.Series(model_df["PowerPlay"].dropna().unique()).sort_values().tolist())


After merge: non-null PowerPlay rows in model_df: 0
Unique PowerPlay values in model_df: []


In [47]:
# Try joining without TeamID just as a test
test_merge = model_df.merge(
    ends[["CompetitionID","SessionID","GameID","EndID","PowerPlay"]],
    on=["CompetitionID","SessionID","GameID","EndID"],
    how="left",
    suffixes=("","_end")
)

print("Non-null PowerPlay when joining WITHOUT TeamID:", test_merge["PowerPlay"].notna().sum())
print("Unique PowerPlay values (no TeamID join):", pd.Series(test_merge["PowerPlay"].dropna().unique()).sort_values().tolist())


Non-null PowerPlay when joining WITHOUT TeamID: 0
Unique PowerPlay values (no TeamID join): []


In [48]:
model_df["pp_side"] = model_df["PowerPlay"].fillna(0).astype(int).clip(0, 2)
model_df["pp_right"] = (model_df["pp_side"] == 1).astype(int)
model_df["pp_left"]  = (model_df["pp_side"] == 2).astype(int)

print(model_df[["pp_side","pp_right","pp_left"]].value_counts().sort_index())


pp_side  pp_right  pp_left
0        0         0          2637
Name: count, dtype: int64


In [49]:
keys_game = ["CompetitionID", "SessionID", "GameID"]

games_in_model = model_df[keys_game].drop_duplicates()
games_in_ends  = ends[keys_game].drop_duplicates()

print("Unique games in model_df:", len(games_in_model))
print("Unique games in ends:", len(games_in_ends))

merged_games = games_in_model.merge(games_in_ends, on=keys_game, how="inner")
print("Games that overlap:", len(merged_games))

# Show a few keys that are in model_df but not ends
only_model = games_in_model.merge(games_in_ends, on=keys_game, how="left", indicator=True)
print("\nExample game keys only in model_df:")
print(only_model[only_model["_merge"]=="left_only"].head(10))


Unique games in model_df: 344
Unique games in ends: 344
Games that overlap: 344

Example game keys only in model_df:
Empty DataFrame
Columns: [CompetitionID, SessionID, GameID, _merge]
Index: []


In [50]:
keys_end = ["CompetitionID", "SessionID", "GameID", "EndID"]

ends_in_model = model_df[keys_end].drop_duplicates()
ends_in_ends  = ends[keys_end].drop_duplicates()

print("Unique ends in model_df:", len(ends_in_model))
print("Unique ends in ends:", len(ends_in_ends))

merged_ends = ends_in_model.merge(ends_in_ends, on=keys_end, how="inner")
print("Ends that overlap:", len(merged_ends))

# If still 0, show a few end keys that exist only in model_df
only_model_ends = ends_in_model.merge(ends_in_ends, on=keys_end, how="left", indicator=True)
print("\nExample end keys only in model_df:")
print(only_model_ends[only_model_ends["_merge"]=="left_only"].head(10))


Unique ends in model_df: 2637
Unique ends in ends: 2637
Ends that overlap: 2637

Example end keys only in model_df:
Empty DataFrame
Columns: [CompetitionID, SessionID, GameID, EndID, _merge]
Index: []


In [51]:
print("model_df CompetitionID unique:", sorted(model_df["CompetitionID"].unique())[:20])
print("ends CompetitionID unique:", sorted(ends["CompetitionID"].unique())[:20])

print("model_df SessionID min/max:", model_df["SessionID"].min(), model_df["SessionID"].max())
print("ends SessionID min/max:", ends["SessionID"].min(), ends["SessionID"].max())

print("model_df GameID min/max:", model_df["GameID"].min(), model_df["GameID"].max())
print("ends GameID min/max:", ends["GameID"].min(), ends["GameID"].max())

print("model_df EndID unique sample:", sorted(model_df["EndID"].unique())[:20])
print("ends EndID unique sample:", sorted(ends["EndID"].unique())[:20])


model_df CompetitionID unique: [np.int64(0), np.int64(22230015), np.int64(23240026), np.int64(24250026)]
ends CompetitionID unique: [np.int64(0), np.int64(22230015), np.int64(23240026), np.int64(24250026)]
model_df SessionID min/max: 1 49
ends SessionID min/max: 1 49
model_df GameID min/max: 1 5
ends GameID min/max: 1 5
model_df EndID unique sample: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
ends EndID unique sample: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [52]:
print("PowerPlay non-null right before merge:", ends["PowerPlay"].notna().sum())
print("PowerPlay uniques right before merge:", ends["PowerPlay"].dropna().unique()[:10])


PowerPlay non-null right before merge: 598
PowerPlay uniques right before merge: [1. 2.]


In [53]:
[c for c in model_df.columns if "PowerPlay" in c]


['PowerPlay_x', 'PowerPlay_y', 'PowerPlay']

In [54]:
pp_tbl = (
    ends[["CompetitionID","SessionID","GameID","EndID","TeamID","PowerPlay"]]
    .copy()
)

print("pp_tbl PowerPlay non-null:", pp_tbl["PowerPlay"].notna().sum())
print(pp_tbl["PowerPlay"].value_counts(dropna=False))


pp_tbl PowerPlay non-null: 598
PowerPlay
NaN    4676
1.0     313
2.0     285
Name: count, dtype: int64


In [58]:
model_df = model_df.merge(
    pp_tbl,
    on=["CompetitionID","SessionID","GameID","EndID","TeamID"],
    how="left",
    validate="1:1"   # each model_df row should match at most one ends row
)

print("After merge PowerPlay non-null:", model_df["PowerPlay"].notna().sum())
print("PowerPlay counts in model_df:\n", model_df["PowerPlay"].value_counts(dropna=False))


MergeError: Passing 'suffixes' which cause duplicate columns {'PowerPlay_x'} is not allowed.

In [59]:
keys = ["CompetitionID", "SessionID", "GameID", "EndID", "TeamID"]

# 1) Make sure ends PowerPlay is numeric (optional but recommended)
ends["PowerPlay"] = pd.to_numeric(ends["PowerPlay"], errors="coerce")

# 2) Build the small lookup table
pp_tbl = ends[keys + ["PowerPlay"]].copy()

# 3) If model_df already has PowerPlay, drop it so we don't collide
if "PowerPlay" in model_df.columns:
    model_df = model_df.drop(columns=["PowerPlay"])

# 4) Merge cleanly
model_df = model_df.merge(pp_tbl, on=keys, how="left", validate="1:1")

# 5) Sanity check
print("After merge PowerPlay non-null:", model_df["PowerPlay"].notna().sum())
print("After merge PowerPlay values:", model_df["PowerPlay"].value_counts(dropna=False))


After merge PowerPlay non-null: 0
After merge PowerPlay values: PowerPlay
NaN    2637
Name: count, dtype: int64


In [100]:
keys = ["CompetitionID", "SessionID", "GameID", "EndID", "TeamID"]

# build pp table
ends["PowerPlay"] = pd.to_numeric(ends["PowerPlay"], errors="coerce")
pp_tbl = ends[keys + ["PowerPlay"]].copy()

print("pp_tbl PowerPlay non-null:", pp_tbl["PowerPlay"].notna().sum())  # should be 598

# how many key tuples overlap?
overlap = model_df[keys].drop_duplicates().merge(
    pp_tbl[keys].drop_duplicates(),
    on=keys,
    how="inner"
)

print("Unique keys in model_df:", model_df[keys].drop_duplicates().shape[0])
print("Unique keys in pp_tbl:", pp_tbl[keys].drop_duplicates().shape[0])
print("Overlapping key tuples:", overlap.shape[0])


pp_tbl PowerPlay non-null: 598
Unique keys in model_df: 2637
Unique keys in pp_tbl: 5274
Overlapping key tuples: 2637


In [101]:
# overlap without TeamID (end-level)
keys_end = ["CompetitionID", "SessionID", "GameID", "EndID"]

overlap_end = model_df[keys_end].drop_duplicates().merge(
    pp_tbl[keys_end].drop_duplicates(),
    on=keys_end,
    how="inner"
)

print("Overlapping end keys (no TeamID):", overlap_end.shape[0])

print("\nTeamID distribution in model_df (top 15):")
print(model_df["TeamID"].value_counts().head(15))

print("\nTeamID distribution in ends (top 15):")
print(ends["TeamID"].value_counts().head(15))


Overlapping end keys (no TeamID): 2637

TeamID distribution in model_df (top 15):
TeamID
17    180
24    176
20    175
10    160
19    155
46    154
18    153
22    150
37    129
14    122
16    112
44    111
43    108
11    103
51    103
Name: count, dtype: Int64

TeamID distribution in ends (top 15):
TeamID
17    339
10    313
20    311
24    311
19    306
18    302
46    300
22    296
37    263
14    247
16    226
43    220
11    219
51    218
44    218
Name: count, dtype: Int64


In [ ]:
keys_end = ["CompetitionID", "SessionID", "GameID", "EndID"]

# Build end-level PowerPlay: if either team row has it, keep it
pp_end = (
    ends[keys_end + ["PowerPlay"]]
      .assign(PowerPlay=pd.to_numeric(ends["PowerPlay"], errors="coerce"))
      .groupby(keys_end, as_index=False)["PowerPlay"]
      .max()
)

print("pp_end non-null:", pp_end["PowerPlay"].notna().sum())
print(pp_end["PowerPlay"].value_counts(dropna=False))

# Merge at end-level (no TeamID)
model_df = model_df.drop(columns=["PowerPlay"], errors="ignore")
model_df = model_df.merge(pp_end, on=keys_end, how="left", validate="m:1")

print("After merge (end-level) PowerPlay non-null:", model_df["PowerPlay"].notna().sum())
print(model_df["PowerPlay"].value_counts(dropna=False))




pp_end non-null: 598
PowerPlay
NaN    2039
1.0     313
2.0     285
Name: count, dtype: int64
After merge (end-level) PowerPlay non-null: 598
PowerPlay
NaN    2039
1.0     313
2.0     285
Name: count, dtype: int64


In [104]:
pp_end = ends[keys_end + ["PowerPlay"]].groupby(keys_end)["PowerPlay"].max()


In [105]:
model_df["pp_side"] = model_df["PowerPlay"].fillna(0).astype(int)
model_df["pp_right"] = (model_df["pp_side"] == 1).astype(int)
model_df["pp_left"]  = (model_df["pp_side"] == 2).astype(int)


In [106]:
print("pp_side value counts:")
print(model_df["pp_side"].value_counts(dropna=False).sort_index())

print("\nUnique pp_side values:", sorted(model_df["pp_side"].unique()))


pp_side value counts:
pp_side
0    2039
1     313
2     285
Name: count, dtype: int64

Unique pp_side values: [np.int64(0), np.int64(1), np.int64(2)]


In [107]:
bad_right = (model_df["pp_right"] != (model_df["pp_side"] == 1).astype(int)).sum()
bad_left  = (model_df["pp_left"]  != (model_df["pp_side"] == 2).astype(int)).sum()
both_on   = ((model_df["pp_right"] == 1) & (model_df["pp_left"] == 1)).sum()

print("Mismatches pp_right:", bad_right)
print("Mismatches pp_left :", bad_left)
print("Rows where both pp_right and pp_left are 1:", both_on)


Mismatches pp_right: 0
Mismatches pp_left : 0
Rows where both pp_right and pp_left are 1: 0


In [108]:
print("Total rows in model_df:", len(model_df))
print("Unique team-end rows (keys):", model_df[["CompetitionID","SessionID","GameID","EndID","TeamID"]].drop_duplicates().shape[0])


Total rows in model_df: 2637
Unique team-end rows (keys): 2637


In [109]:
keys_end = ["CompetitionID","SessionID","GameID","EndID"]

end_pp_unique_counts = model_df.groupby(keys_end)["pp_side"].nunique()
print("Ends where pp_side differs between teams (should be 0):", (end_pp_unique_counts > 1).sum())

# Optional: show a few problematic ends if any exist
if (end_pp_unique_counts > 1).any():
    print("\nExample ends with mismatch:")
    print(end_pp_unique_counts[end_pp_unique_counts > 1].head(10))


Ends where pp_side differs between teams (should be 0): 0


In [110]:
print(model_df.groupby("pp_side")["Result_binned"].value_counts(normalize=True).unstack(fill_value=0))


Result_binned         0         1         2         3
pp_side                                              
0              0.663561  0.219225  0.067190  0.050025
1              0.801917  0.121406  0.044728  0.031949
2              0.754386  0.150877  0.056140  0.038596


Now were going to attempt to attach the lsfe, who had the last stone first end to the that data frame. Dont exactly know how to do this but we'll see

In [ ]:

games_keys = ["CompetitionID", "SessionID", "GameID"]
games_lsfe = games[games_keys + ["LSFE", "TeamID1", "TeamID2"]].copy()

model_df = model_df.merge(
    games_lsfe,
    on=games_keys,
    how="left",
    validate="m:1"
)
# NICEEE!! THIS TOOK SO LONG

This is just alternating hammer every end

In [114]:
# End numbering starts at 1
model_df["end_parity"] = (model_df["EndID"] % 2).astype(int)


In [118]:
print("TeamID1_x == TeamID1_y mismatches:", (model_df["TeamID1_x"] != model_df["TeamID1_y"]).sum())
print("TeamID2_x == TeamID2_y mismatches:", (model_df["TeamID2_x"] != model_df["TeamID2_y"]).sum())


TeamID1_x == TeamID1_y mismatches: 0
TeamID2_x == TeamID2_y mismatches: 0


In [119]:
# Create canonical columns (prefer _x)
model_df["TeamID1"] = model_df["TeamID1_x"]
model_df["TeamID2"] = model_df["TeamID2_x"]

# Drop the messy duplicates so this never bites you again
model_df = model_df.drop(columns=["TeamID1_x","TeamID2_x","TeamID1_y","TeamID2_y"])


In [120]:
print({c: (c in model_df.columns) for c in ["LSFE","TeamID1","TeamID2","end_parity"]})
print([c for c in model_df.columns if "TeamID" in c or "LSFE" in c])


{'LSFE': True, 'TeamID1': True, 'TeamID2': True, 'end_parity': True}
['TeamID', 'LSFE', 'TeamID1', 'TeamID2']


After fixing some logic issues with teamID's were going to try to redo hammer logic

In [ ]:

model_df["end_parity"] = (model_df["EndID"] % 2).astype(int)


end1_hammer_team = model_df["TeamID1"].where(model_df["LSFE"] == 1, model_df["TeamID2"])


other_team = model_df["TeamID2"].where(model_df["LSFE"] == 1, model_df["TeamID1"])


hammer_team = end1_hammer_team.where(model_df["end_parity"] == 1, other_team)


model_df["hammer"] = (model_df["TeamID"] == hammer_team).astype(int)


In [124]:
end_sizes = model_df.groupby(["CompetitionID","SessionID","GameID","EndID"]).size()
print(end_sizes.value_counts().sort_index().head(10))
print("Ends with size != 1:", (end_sizes != 1).sum())


1    2637
Name: count, dtype: int64
Ends with size != 1: 0


In [125]:
print(model_df["hammer"].isna().sum())
print(model_df["hammer"].value_counts(dropna=False))


0
hammer
0    1627
1    1010
Name: count, dtype: int64


In [126]:
bad = ~model_df["TeamID"].isin(model_df["TeamID1"].tolist()) & ~model_df["TeamID"].isin(model_df["TeamID2"].tolist())
# better version row-wise:
bad = ~((model_df["TeamID"] == model_df["TeamID1"]) | (model_df["TeamID"] == model_df["TeamID2"]))
print("Rows where TeamID not in {TeamID1, TeamID2}:", bad.sum())


Rows where TeamID not in {TeamID1, TeamID2}: 0


In [135]:
[c for c in model_df.columns if "TeamID" in c]


['TeamID', 'TeamID1', 'TeamID2']

In [136]:
team_cols = [c for c in ["TeamID1_x","TeamID1_y","TeamID2_x","TeamID2_y"] if c in model_df.columns]
print("Found these duplicate cols:", team_cols)
print("Null counts:\n", model_df[team_cols].isna().sum())


Found these duplicate cols: []
Null counts:
 Series([], dtype: float64)


In [137]:
[c for c in model_df.columns if "TeamID" in c]


['TeamID', 'TeamID1', 'TeamID2']

OK NOW we can add logic to who had the hammer in a certain row. This is just fliiping hammer ownership by who was thowing last end.

In [138]:
model_df["hammer"] = 0

# End 1 hammer holder:
# If LSFE==1 => TeamID1 has hammer in End 1
# If LSFE==0 => TeamID2 has hammer in End 1
is_end1 = (model_df["end_parity"] == 1)

model_df.loc[is_end1 & (model_df["LSFE"] == 1) & (model_df["TeamID"] == model_df["TeamID1"]), "hammer"] = 1
model_df.loc[is_end1 & (model_df["LSFE"] == 0) & (model_df["TeamID"] == model_df["TeamID2"]), "hammer"] = 1

In [139]:
# Flip hammer on even ends
is_even_end = (model_df["end_parity"] == 0)
model_df.loc[is_even_end, "hammer"] = 1 - model_df.loc[is_even_end, "hammer"]


In [140]:
hammer_sum = model_df.groupby(["CompetitionID","SessionID","GameID","EndID"])["hammer"].sum()
print("Ends with hammer != 1:", (hammer_sum != 1).sum())
print(hammer_sum.value_counts().sort_index())


Ends with hammer != 1: 891
hammer
0     891
1    1746
Name: count, dtype: int64


Here were are testing are distriubutions from 1';s and 0's for who has the hammer

In [141]:
print(model_df["hammer"].value_counts(normalize=True))


hammer
1    0.662116
0    0.337884
Name: proportion, dtype: float64


check if every rows temid matchs teamid1 or teamid2

In [142]:
mask_valid_team = (model_df["TeamID"] == model_df["TeamID1"]) | (model_df["TeamID"] == model_df["TeamID2"])
print("Rows where TeamID is neither TeamID1 nor TeamID2:", (~mask_valid_team).sum())


Rows where TeamID is neither TeamID1 nor TeamID2: 0


In [143]:
lsfe_by_game = model_df.groupby(["CompetitionID","SessionID","GameID"])["LSFE"].nunique(dropna=False)
print(lsfe_by_game.value_counts().sort_index())


LSFE
1    344
Name: count, dtype: int64


In [144]:
game_keys = ["CompetitionID","SessionID","GameID"]

# Determine which team has LSFE per game:
# LSFE==1 TeamID1 has hammer in end 1
# LSFE==0 TeamID2 has hammer in end 1
game_lsfe = (
    model_df[game_keys + ["TeamID1","TeamID2","LSFE"]]
    .drop_duplicates(subset=game_keys)  # 1 row per game
    .copy()
)

game_lsfe["hammer_end1_team"] = game_lsfe.apply(
    lambda r: r["TeamID1"] if r["LSFE"] == 1 else r["TeamID2"],
    axis=1
)


In [145]:
model_df = model_df.merge(game_lsfe[game_keys + ["hammer_end1_team"]], on=game_keys, how="left", validate="m:1")

# baseline alternating hammer: odd end => hammer_end1_team, even end => other team
is_odd_end = (model_df["EndID"] % 2 == 1)

# Determine the "other team" for each row
model_df["other_team"] = model_df["TeamID1"].where(model_df["hammer_end1_team"] != model_df["TeamID1"], model_df["TeamID2"])

# Which team should have hammer in this end
model_df["hammer_team_this_end"] = model_df["hammer_end1_team"].where(is_odd_end, model_df["other_team"])

# Finally hammer flag for this row
model_df["hammer"] = (model_df["TeamID"] == model_df["hammer_team_this_end"]).astype(int)


In [146]:
hammer_sum = model_df.groupby(["CompetitionID","SessionID","GameID","EndID"])["hammer"].sum()
print("Ends with hammer != 1:", (hammer_sum != 1).sum())
print(hammer_sum.value_counts().sort_index())
print(model_df["hammer"].value_counts(normalize=True))


Ends with hammer != 1: 1627
hammer
0    1627
1    1010
Name: count, dtype: int64
hammer
0    0.616989
1    0.383011
Name: proportion, dtype: float64


In [150]:
model_df["hammer"].value_counts(dropna=False)


hammer
0    1627
1    1010
Name: count, dtype: int64

In [154]:
[c for c in model_df.columns if "hammer" in c.lower()]


['hammer', 'hammer_end1_team', 'hammer_team_this_end']

In [155]:
print(model_df["hammer"].value_counts(dropna=False))


hammer
0    1627
1    1010
Name: count, dtype: int64


In [156]:
bad = ~model_df["hammer_team_this_end"].isin([model_df["TeamID1"], model_df["TeamID2"]])
print("Bad hammer_team_this_end rows:", bad.sum())


Bad hammer_team_this_end rows: 2637


In [157]:
agree = (model_df["hammer"] == (model_df["TeamID"] == model_df["hammer_team_this_end"]).astype(int)).mean()
print("Hammer agrees with hammer_team_this_end:", agree)


Hammer agrees with hammer_team_this_end: 1.0


In [158]:
end_keys = ["CompetitionID","SessionID","GameID","EndID"]
hammer_sum = model_df.groupby(end_keys)["hammer"].sum()
print("Ends where hammer_sum != 1:", (hammer_sum != 1).sum())
print(hammer_sum.value_counts().sort_index().head(10))


Ends where hammer_sum != 1: 1627
hammer
0    1627
1    1010
Name: count, dtype: int64


In [159]:
end_keys = ["CompetitionID", "SessionID", "GameID", "EndID"]

# 1) Collapse to ONE row per end (keep the first TeamID1/TeamID2 you see)
ends_base = (
    model_df[end_keys + ["TeamID1", "TeamID2", "LSFE"]]
    .drop_duplicates(end_keys)
    .copy()
)

print("ends_base rows:", len(ends_base))
print("unique ends:", model_df[end_keys].drop_duplicates().shape[0])


ends_base rows: 2637
unique ends: 2637


In [160]:
# Make sure end_parity exists at end-level (if you already have it per row, collapse it)
if "end_parity" in model_df.columns:
    end_parity_tbl = model_df[end_keys + ["end_parity"]].drop_duplicates(end_keys)
    ends_base = ends_base.merge(end_parity_tbl, on=end_keys, how="left", validate="1:1")
else:
    # fallback: derive from EndID (1=odd, 0=even)
    ends_base["end_parity"] = (ends_base["EndID"] % 2).astype(int)

# Bring hammer_end1_team (should be constant within an end; if not, we’ll fix later)
hammer1_tbl = model_df[end_keys + ["hammer_end1_team"]].drop_duplicates(end_keys)
ends_base = ends_base.merge(hammer1_tbl, on=end_keys, how="left", validate="1:1")

# Hammer team this end: flip on even parity
ends_base["hammer_team_this_end"] = ends_base["hammer_end1_team"]
ends_base.loc[ends_base["end_parity"] == 0, "hammer_team_this_end"] = (
    ends_base.loc[ends_base["end_parity"] == 0, "TeamID1"]
    + ends_base.loc[ends_base["end_parity"] == 0, "TeamID2"]
    - ends_base.loc[ends_base["end_parity"] == 0, "hammer_end1_team"]
)


This validates that the ahmmer system is working and has been set up correctly to start training

In [166]:
# 1) hammer_team_this_end must be either TeamID1 or TeamID2 (row-wise check)
bad = ~((ends_base["hammer_team_this_end"] == ends_base["TeamID1"]) |
        (ends_base["hammer_team_this_end"] == ends_base["TeamID2"]))
print("Bad hammer_team_this_end rows:", bad.sum())

# 2) Hammer one-hot indicators (competition-friendly)
ends_base["hammer_is_team1"] = (ends_base["hammer_team_this_end"] == ends_base["TeamID1"]).astype(int)
ends_base["hammer_is_team2"] = (ends_base["hammer_team_this_end"] == ends_base["TeamID2"]).astype(int)

print("hammer_is_team1 value counts:", ends_base["hammer_is_team1"].value_counts())
print("hammer_is_team2 value counts:", ends_base["hammer_is_team2"].value_counts())

# 3) Exactly one of the two should be 1 each end
print("Ends where hammer_is_team1 + hammer_is_team2 != 1:",
      ((ends_base["hammer_is_team1"] + ends_base["hammer_is_team2"]) != 1).sum())


Bad hammer_team_this_end rows: 0
hammer_is_team1 value counts: hammer_is_team1
1    1322
0    1315
Name: count, dtype: int64
hammer_is_team2 value counts: hammer_is_team2
0    1322
1    1315
Name: count, dtype: int64
Ends where hammer_is_team1 + hammer_is_team2 != 1: 0


In [165]:
# Drop any older hammer columns to avoid confusion
model_df = model_df.drop(columns=["hammer", "hammer_team_this_end"], errors="ignore")

model_df = model_df.merge(
    ends_base[end_keys + ["hammer_team_this_end", "hammer_is_team1", "hammer_is_team2"]],
    on=end_keys,
    how="left",
    validate="m:1"
)

# team-row hammer flag: 1 if this row's TeamID is the hammer team
model_df["hammer"] = (model_df["TeamID"] == model_df["hammer_team_this_end"]).astype(int)


Next well start actually training and using the xgboost model. first well focus on quantifying how having the hammer impacts expected win percentage and expected points in cetain scenarios.

In [167]:
import re

# 1) Required columns for end-level modeling
required = ["CompetitionID","SessionID","GameID","EndID","TeamID1","TeamID2","hammer_team_this_end"]
missing = [c for c in required if c not in ends_base.columns]
print("Missing required columns:", missing)

# 2) Identify possible scoring columns (you choose from these)
score_like = []
for c in ends_base.columns:
    if re.search(r"(score|points|pts|scored|hammer_score|end_score)", c, re.IGNORECASE):
        score_like.append(c)

print("\nScore-like columns (candidates):")
print(score_like[:60])  # print first 60 candidates
print("\nTotal columns:", len(ends_base.columns))


Missing required columns: []

Score-like columns (candidates):
[]

Total columns: 12


## XGBoost Readiness Assessment

This cell checks if `model_df` meets all the criteria necessary to start XGBoost modeling:

**Required Criteria:**
1. ✅ **Target Variable**: Must have a target to predict (e.g., `Result_binned`, `y_hammer_scored`)
2. ✅ **Features**: Must have feature columns (numeric or categorical)
3. ✅ **No Data Leakage**: Features shouldn't contain future information
4. ✅ **Appropriate Data Types**: Numeric or properly encoded categorical
5. ✅ **Missing Values**: Should be handled (XGBoost can handle some, but excessive missing is problematic)
6. ✅ **Sufficient Data**: Enough rows for train/test split

**What XGBoost Needs:**
- **X (features)**: Array/DataFrame of numeric or encoded categorical features
- **y (target)**: Array/Series of target values
- **Objective function**: 
  - `multi:softprob` for multi-class classification (Result_binned: 0,1,2,3)
  - `binary:logistic` for binary classification (hammer scored: 0,1)
- **Train/test split**: Should group by GameID to avoid data leakage


In [3]:
# ============================================
# XGBOOST READINESS ASSESSMENT
# ============================================
# Check if model_df meets criteria for XGBoost modeling

print("=" * 60)
print("XGBOOST MODELING READINESS CHECK")
print("=" * 60)

# 1. Check if model_df exists and has data
if 'model_df' not in locals() and 'model_df' not in globals():
    print("❌ CRITICAL: model_df does not exist!")
else:
    print(f"✅ model_df exists with shape: {model_df.shape}")
    print(f"   Rows: {len(model_df)}, Columns: {len(model_df.columns)}")
    
    # 2. Check for target variable
    print("\n" + "-" * 60)
    print("TARGET VARIABLE CHECK:")
    print("-" * 60)
    
    target_candidates = ['Result_binned', 'Result', 'y_hammer_scored']
    found_targets = [t for t in target_candidates if t in model_df.columns]
    
    if found_targets:
        print(f"✅ Found target variable(s): {found_targets}")
        for target in found_targets:
            print(f"\n   {target}:")
            print(f"   - Missing values: {model_df[target].isna().sum()} ({model_df[target].isna().mean()*100:.1f}%)")
            print(f"   - Unique values: {sorted(model_df[target].dropna().unique())}")
            print(f"   - Value counts:")
            print(model_df[target].value_counts().sort_index())
    else:
        print("❌ CRITICAL: No target variable found!")
        print(f"   Available columns: {list(model_df.columns)[:20]}...")
    
    # 3. Identify feature columns
    print("\n" + "-" * 60)
    print("FEATURE COLUMNS CHECK:")
    print("-" * 60)
    
    # Columns to exclude (keys, targets, identifiers)
    exclude_cols = [
        'CompetitionID', 'SessionID', 'GameID', 'EndID', 'TeamID', 
        'TeamID1', 'TeamID2', 'PlayerID', 'ShotID', 'shot_order',
        'Result', 'Result_binned', 'y_hammer_scored',
        'hammer_team_this_end', 'hammer_end1_team', 'other_team',
        'scoring_team', 'end_points_team1', 'end_points_team2'
    ]
    
    # Only exclude columns that actually exist
    exclude_cols = [c for c in exclude_cols if c in model_df.columns]
    
    # Feature columns = all columns except excluded ones
    feature_cols = [c for c in model_df.columns if c not in exclude_cols]
    
    print(f"✅ Found {len(feature_cols)} potential feature columns")
    print(f"   Excluded {len(exclude_cols)} key/target columns")
    print(f"\n   Feature columns:")
    for i, col in enumerate(feature_cols, 1):
        dtype = model_df[col].dtype
        missing = model_df[col].isna().sum()
        missing_pct = (missing / len(model_df)) * 100
        print(f"   {i:2d}. {col:30s} | dtype: {str(dtype):10s} | missing: {missing:4d} ({missing_pct:5.1f}%)")
    
    # 4. Check data types for XGBoost compatibility
    print("\n" + "-" * 60)
    print("DATA TYPE COMPATIBILITY:")
    print("-" * 60)
    
    numeric_features = []
    categorical_features = []
    problematic_features = []
    
    for col in feature_cols:
        dtype = model_df[col].dtype
        if pd.api.types.is_numeric_dtype(dtype):
            numeric_features.append(col)
        elif pd.api.types.is_object_dtype(dtype) or pd.api.types.is_string_dtype(dtype):
            categorical_features.append(col)
        else:
            problematic_features.append((col, dtype))
    
    print(f"✅ Numeric features: {len(numeric_features)}")
    print(f"⚠️  Categorical features: {len(categorical_features)}")
    if categorical_features:
        print(f"   Note: XGBoost can handle categoricals, but may need encoding")
        print(f"   Categorical columns: {categorical_features[:10]}")
    if problematic_features:
        print(f"❌ Problematic features: {problematic_features}")
    
    # 5. Check for missing values in features
    print("\n" + "-" * 60)
    print("MISSING VALUES ANALYSIS:")
    print("-" * 60)
    
    features_with_missing = []
    for col in feature_cols:
        missing = model_df[col].isna().sum()
        if missing > 0:
            features_with_missing.append((col, missing, (missing/len(model_df))*100))
    
    if features_with_missing:
        print(f"⚠️  {len(features_with_missing)} features have missing values:")
        for col, missing, pct in sorted(features_with_missing, key=lambda x: x[1], reverse=True)[:10]:
            print(f"   {col:30s}: {missing:4d} ({pct:5.1f}%)")
        print("   Note: XGBoost handles missing values, but review if excessive")
    else:
        print("✅ No missing values in feature columns")
    
    # 6. Check for potential data leakage
    print("\n" + "-" * 60)
    print("DATA LEAKAGE CHECK:")
    print("-" * 60)
    
    leakage_keywords = ['result', 'score', 'points', 'outcome', 'win', 'loss']
    potential_leakage = []
    for col in feature_cols:
        col_lower = col.lower()
        if any(keyword in col_lower for keyword in leakage_keywords):
            potential_leakage.append(col)
    
    if potential_leakage:
        print(f"⚠️  Potential leakage columns (review carefully):")
        for col in potential_leakage:
            print(f"   - {col}")
    else:
        print("✅ No obvious leakage columns detected")
    
    # 7. Summary and recommendations
    print("\n" + "=" * 60)
    print("SUMMARY & RECOMMENDATIONS:")
    print("=" * 60)
    
    issues = []
    if not found_targets:
        issues.append("❌ Missing target variable")
    if len(feature_cols) == 0:
        issues.append("❌ No feature columns identified")
    if len(numeric_features) == 0 and len(categorical_features) == 0:
        issues.append("❌ No usable features found")
    
    if issues:
        print("CRITICAL ISSUES FOUND:")
        for issue in issues:
            print(f"   {issue}")
        print("\n   ACTION REQUIRED: Fix issues before proceeding with XGBoost")
    else:
        print("✅ BASIC REQUIREMENTS MET:")
        print(f"   - Target variable: {found_targets[0] if found_targets else 'None'}")
        print(f"   - Feature count: {len(feature_cols)}")
        print(f"   - Numeric features: {len(numeric_features)}")
        print(f"   - Data shape: {model_df.shape}")
        
        print("\n   NEXT STEPS:")
        print("   1. Handle missing values (if any)")
        print("   2. Encode categorical features (if any)")
        print("   3. Split into train/test sets (group by GameID to avoid leakage)")
        print("   4. Set up XGBoost with appropriate objective:")
        if 'Result_binned' in found_targets:
            print("      - multi:softprob (for 4-class classification: 0,1,2,3)")
        elif 'y_hammer_scored' in found_targets:
            print("      - binary:logistic (for binary classification)")
        print("   5. Configure class weights (if imbalanced)")
        
        print("\n   READY FOR MODELING: YES ✅")
    
    print("\n" + "=" * 60)


XGBOOST MODELING READINESS CHECK
❌ CRITICAL: model_df does not exist!


In [169]:
# --- YOU MUST SET THESE TWO AFTER LOOKING AT CHUNK 1 OUTPUT ---
team1_points_col = "PUT_TEAM1_POINTS_COL_HERE"
team2_points_col = "PUT_TEAM2_POINTS_COL_HERE"

# Coerce to numeric safely
ends_base[team1_points_col] = pd.to_numeric(ends_base[team1_points_col], errors="coerce")
ends_base[team2_points_col] = pd.to_numeric(ends_base[team2_points_col], errors="coerce")

# Did each team score this end?
ends_base["team1_scored"] = (ends_base[team1_points_col] > 0).astype(int)
ends_base["team2_scored"] = (ends_base[team2_points_col] > 0).astype(int)

# Hammer team scored?
ends_base["y_hammer_scored"] = (
    ((ends_base["hammer_team_this_end"] == ends_base["TeamID1"]) & (ends_base["team1_scored"] == 1)) |
    ((ends_base["hammer_team_this_end"] == ends_base["TeamID2"]) & (ends_base["team2_scored"] == 1))
).astype(int)

print("Label balance (y_hammer_scored):")
print(ends_base["y_hammer_scored"].value_counts(dropna=False))
print("Positive rate:", ends_base["y_hammer_scored"].mean())


KeyError: 'PUT_TEAM1_POINTS_COL_HERE'